<a href="https://colab.research.google.com/github/virexan/DPA-EfficientNet_model/blob/main/DPA_EFFIECIENET_SKIN_CANCER_MODEL_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ── Install / verify packages ──────────────────────────────────────────────
!pip install -q kaggle timm grad-cam

import os, shutil, glob, warnings
warnings.filterwarnings('ignore')

# ── Kaggle credentials (upload kaggle.json once) ───────────────────────────
from google.colab import files
print('📂 Upload your kaggle.json file:')
uploaded = files.upload()          # ONE-TIME upload

os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
shutil.move('kaggle.json', os.path.expanduser('~/.kaggle/kaggle.json'))
os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)

# ── Download HAM10000 ──────────────────────────────────────────────────────
print('\n⬇️  Downloading HAM10000 dataset...')
!kaggle datasets download -d kmader/skin-cancer-mnist-ham10000 -q
!unzip -q skin-cancer-mnist-ham10000.zip -d dataset

print('\n✅ Dataset ready.')
!find dataset -name '*.jpg' | wc -l{INSPECT THE CODE N GIVE THE MODIFIED VERSION AS A LOT OF ERRORS ARE HAPPENING}

In [ ]:
# ── Core imports ───────────────────────────────────────────────────────────
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision.models as tv_models
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.cuda.amp import GradScaler, autocast

import pandas as pd, numpy as np, cv2, os, glob, time, copy
import matplotlib.pyplot as plt, matplotlib.patches as mpatches
import seaborn as sns
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, roc_auc_score, roc_curve, auc
)
from sklearn.preprocessing import label_binarize
from sklearn.utils.class_weight import compute_class_weight

# ── Global configuration  ──────────────────────────────────────────────────────────
SEED        = 42
IMG_SIZE    = 224
BATCH_SIZE  = 32
NUM_CLASSES = 7
DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

CLASS_NAMES = ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']
CLASS_FULL  = [
    'Actinic Keratosis', 'Basal Cell Carcinoma', 'Benign Keratosis',
    'Dermatofibroma', 'Melanoma', 'Melanocytic Nevi', 'Vascular Lesion'
]

torch.manual_seed(SEED)
np.random.seed(SEED)
print(f'✅ Config ready  |  Device: {DEVICE}  |  Classes: {NUM_CLASSES}')

In [ ]:
# ==========================================================
# Download and Extract ISIC 2019 Dataset
# ==========================================================

import os
import zipfile
from glob import glob

# Upgrade Kaggle (optional but recommended)
!pip install -q --upgrade kaggle

print("📥 Downloading ISIC 2019 dataset from Kaggle...")

# Download dataset
!kaggle datasets download -d andrewmvd/isic-2019 --force

# Create extraction folder
os.makedirs("/content/isic2019", exist_ok=True)

print("\n📦 Extracting dataset...")

# Extract
with zipfile.ZipFile("isic-2019.zip", "r") as zip_ref:
    zip_ref.extractall("/content/isic2019")

print("\n🔍 Locating image folder...")

# Find the folder containing JPG images
ISIC_DIR = None
for root, dirs, files in os.walk("/content/isic2019"):
    if any(f.lower().endswith(".jpg") for f in files):
        ISIC_DIR = root
        break

if ISIC_DIR is None:
    raise Exception("❌ Could not find ISIC image directory!")

num_images = len(glob(os.path.join(ISIC_DIR, "*.jpg")))

print("=" * 60)
print(f"✅ ISIC Image Folder : {ISIC_DIR}")
print(f"✅ Total ISIC Images : {num_images}")
print("=" * 60)

In [ ]:
# ==========================================================
# Duplicate Detection using Perceptual Hash (Fast Version)
# ==========================================================

!pip install -q imagehash

import os
from PIL import Image
import imagehash
from tqdm import tqdm
import pandas as pd
from glob import glob

# ---------- Load image paths ----------
ham_images = []

HAM_DIRS = [
    "/content/dataset/HAM10000_images_part_1",
    "/content/dataset/HAM10000_images_part_2"
]

for folder in HAM_DIRS:
    ham_images.extend(glob(folder + "/*.jpg"))

isic_images = glob(ISIC_DIR + "/*.jpg")

print(f"HAM10000 Images : {len(ham_images)}")
print(f"ISIC2019 Images : {len(isic_images)}")

# ---------- Compute hashes ----------

ham_hashes = {}

print("\nComputing HAM10000 hashes...")

for img in tqdm(ham_images):
    try:
        ham_hashes[os.path.basename(img)] = imagehash.phash(Image.open(img).convert("RGB"))
    except:
        pass

isic_hashes = {}

print("\nComputing ISIC2019 hashes...")

for img in tqdm(isic_images):
    try:
        isic_hashes[os.path.basename(img)] = imagehash.phash(Image.open(img).convert("RGB"))
    except:
        pass

# ---------- Fast duplicate detection ----------

duplicates = []

for ham_name, ham_hash in tqdm(ham_hashes.items()):

    best_distance = 100
    best_match = None

    for isic_name, isic_hash in isic_hashes.items():

        distance = ham_hash - isic_hash

        if distance < best_distance:
            best_distance = distance
            best_match = isic_name

    if best_distance <= 1:      # Only exact or almost exact duplicates
        duplicates.append({
            "HAM10000": ham_name,
            "ISIC2019": best_match,
            "Distance": best_distance
        })

dup_df = pd.DataFrame(duplicates)

print("Duplicates:", len(dup_df))

In [ ]:
# %% ─── CELL 4: Grad-CAM++ ────────────────────────────────────────────────────
class GradCAMPlusPlus:
    """
    Grad-CAM++ implementation for CNN explainability.
    Hooks into the target layer to capture forward activations and backward gradients.
    """
    def __init__(self, model, target_layer):
        print("Initializing GradCAMPlusPlus with corrected hooks.") # Added for debugging
        self.model = model
        self.gradients = None
        self.activations = None
        self._hooks = [
            target_layer.register_forward_hook(self._save_activation),
            target_layer.register_backward_hook(self._save_gradient), # Confirmed change to register_backward_hook
        ]

    def _save_activation(self, module, input, output):
        self.activations = output

    def _save_gradient(self, module, grad_in, grad_out):
        self.gradients = grad_out[0]

    def remove_hooks(self):
        for h in self._hooks:
            h.remove()

    def __call__(self, x):
        self.model.eval()
        logits = self.model(x)
        class_idx = torch.argmax(logits, dim=1).item()

        self.model.zero_grad()
        logits[0, class_idx].backward()

        grads = self.gradients       # [1, C, H, W]
        acts  = self.activations     # [1, C, H, W]
        b, k, u, v = grads.size()

        # Grad-CAM++ weight calculation
        alpha_num   = grads.pow(2)
        alpha_denom = 2 * grads.pow(2) + (acts * grads.pow(3)).sum(dim=(2, 3), keepdim=True)
        alpha_denom = torch.where(alpha_denom != 0, alpha_denom, torch.ones_like(alpha_denom))
        alphas  = alpha_num / alpha_denom
        weights = torch.max(logits[0, class_idx].exp() * alphas * F.relu(grads), dim=1)[0]

        cam = F.relu((weights * acts).sum(dim=1, keepdim=True))
        cam = cam.view(b, -1)
        cam -= cam.min(dim=1, keepdim=True)[0]
        cam /= (cam.max(dim=1, keepdim=True)[0] + 1e-8)
        cam = cam.view(b, u, v)

        return cam.detach().cpu().numpy(), class_idx, logits.detach().cpu()


In [ ]:
import pandas as pd

ham = pd.read_csv("/content/dataset/HAM10000_metadata.csv")

print(ham.columns.tolist())
print(ham.head())

In [ ]:
import os
import pandas as pd

for root, dirs, files in os.walk("/content/isic2019"):
    for f in files:
        if "Metadata" in f and f.endswith(".csv"):
            path = os.path.join(root, f)
            print(path)

isic = pd.read_csv(path)

print(isic.columns.tolist())
print(isic.head())

In [ ]:
# ── Load metadata ──────────────────────────────────────────────────────────
df = pd.read_csv('dataset/HAM10000_metadata.csv')

# Map image_id → file path
path_map = {
    os.path.splitext(os.path.basename(p))[0]: p
    for p in glob.glob('dataset/**/*.jpg', recursive=True)
}
df['path'] = df['image_id'].map(path_map)
df = df.dropna(subset=['path']).reset_index(drop=True)
df['label'] = df['dx'].map({c: i for i, c in enumerate(CLASS_NAMES)})

# ── Patient-level split (Section 3.1 — prevents data leakage) ─────────────
unique_patients = df['lesion_id'].unique()
train_pat, val_pat = train_test_split(unique_patients, test_size=0.20, random_state=SEED)
train_df = df[df['lesion_id'].isin(train_pat)].reset_index(drop=True)
val_df   = df[df['lesion_id'].isin(val_pat)].reset_index(drop=True)

print(f'Train: {len(train_df)} images  |  Val: {len(val_df)} images')
print(f'Class distribution (train):\n{train_df["dx"].value_counts()}\n')

# ── Class distribution plot ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
counts = train_df['dx'].value_counts()
colors = ['#e74c3c' if c == 'mel' else '#3498db' for c in counts.index]

axes[0].bar([CLASS_FULL[CLASS_NAMES.index(c)] for c in counts.index],
            counts.values, color=colors)
axes[0].set_title('HAM10000 — Training Set Distribution', fontweight='bold')
axes[0].set_ylabel('Images')
axes[0].tick_params(axis='x', rotation=35)
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 15, str(v), ha='center', fontsize=9)

axes[1].pie(counts.values,
            labels=[CLASS_NAMES[CLASS_NAMES.index(c)] for c in counts.index],
            autopct='%1.1f%%', startangle=140)
axes[1].set_title('Class Proportions', fontweight='bold')
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Transforms (Section 3.1) ───────────────────────────────────────────────
MEAN = [0.485, 0.456, 0.406]   # ImageNet stats (Eq. 1)
STD  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD)
])

val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD)
])


# ── Custom Dataset with hair-artifact removal (DullRazor-inspired) ─────────
class HAMDataset(Dataset):
    def __init__(self, df, transform=None, apply_hair_removal=True):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.apply_hair_removal = apply_hair_removal

    def _remove_hair(self, img_np):
        """Median filtering for hair artifact removal (Section 3.1)."""
        return cv2.medianBlur(img_np, 5)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['path']).convert('RGB')
        if self.apply_hair_removal:
            img_np = np.array(img, dtype=np.uint8)
            img_np = self._remove_hair(img_np)
            img = Image.fromarray(img_np)
        if self.transform:
            img = self.transform(img)
        return img, int(row['label'])


# ── Weighted sampler to handle class imbalance ─────────────────────────────
train_labels = train_df['label'].values
class_weights = compute_class_weight('balanced', classes=np.arange(NUM_CLASSES), y=train_labels)
sample_weights = torch.tensor([class_weights[l] for l in train_labels], dtype=torch.float)
sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

train_ds = HAMDataset(train_df, transform=train_tf)
val_ds   = HAMDataset(val_df,   transform=val_tf, apply_hair_removal=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f'✅ DataLoaders ready')
print(f'   Train batches: {len(train_loader)}  |  Val batches: {len(val_loader)}')
print(f'   Class weights: {dict(zip(CLASS_NAMES, class_weights.round(3)))}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  Section 3.3 — Spatial Attention Module
# ══════════════════════════════════════════════════════════════════════════
class SpatialAttention(nn.Module):
    """
    A_s = σ( f^{7×7}( [AvgPool(F) ; MaxPool(F)] ) )   -- Eq. 3
    F_s = A_s ⊙ F                                      -- Eq. 4
    """
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv    = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)        # (B, 1, H, W)
        max_out, _ = torch.max(x, dim=1, keepdim=True)      # (B, 1, H, W)
        attn = self.sigmoid(self.conv(torch.cat([avg_out, max_out], dim=1)))
        return x * attn                                      # Hadamard product


# ══════════════════════════════════════════════════════════════════════════
#  Proposed: DPA-EfficientNet (Sections 3.2 – 3.5)
# ══════════════════════════════════════════════════════════════════════════
class DPA_EfficientNet(nn.Module):
    """
    EfficientNet-B0 → Spatial Attention → GAP + GMP → FC(512) → FC(7)
    Dual-pooled vector z = [z_avg ; z_max] ∈ R^{2C}   -- Eq. 7
    """
    def __init__(self, num_classes=NUM_CLASSES, dropout=0.3):
        super().__init__()
        weights         = tv_models.EfficientNet_B0_Weights.DEFAULT
        backbone        = tv_models.efficientnet_b0(weights=weights)
        self.features   = backbone.features     # Output: (B, 1280, 7, 7)
        self.spatial_attn = SpatialAttention(kernel_size=7)
        self.classifier = nn.Sequential(
            nn.Linear(1280 * 2, 512),           # Eq. 8: h = δ(W1·z + b1)
            nn.ReLU(),
            nn.Dropout(p=dropout),
            nn.Linear(512, num_classes)         # Eq. 9: y = Softmax(W2·h + b2)
        )

    def forward(self, x):
        F_map = self.features(x)                            # Eq. 2
        F_s   = self.spatial_attn(F_map)                   # Eq. 4
        z_avg = F_s.mean(dim=[2, 3])                       # Eq. 5: GAP
        z_max = F_s.flatten(2).max(dim=2)[0]              # Eq. 6: GMP
        z     = torch.cat([z_avg, z_max], dim=1)           # Eq. 7
        return self.classifier(z)


# ══════════════════════════════════════════════════════════════════════════
#  Baseline model factory
# ══════════════════════════════════════════════════════════════════════════
def build_baseline(name, num_classes=NUM_CLASSES, freeze_backbone=False):
    """
    Builds MobileNetV2 / ResNet50 / EfficientNetB0 with standard GAP head.
    freeze_backbone=False → full fine-tuning (fair comparison).
    """
    if name == 'MobileNetV2':
        m = tv_models.mobilenet_v2(weights=tv_models.MobileNet_V2_Weights.DEFAULT)
        if freeze_backbone:
            for p in m.features.parameters(): p.requires_grad_(False)
        m.classifier = nn.Sequential(
            nn.Dropout(0.2),
            nn.Linear(m.last_channel, num_classes)
        )
    elif name == 'ResNet50':
        m = tv_models.resnet50(weights=tv_models.ResNet50_Weights.DEFAULT)
        if freeze_backbone:
            for name_, p in m.named_parameters():
                if 'layer4' not in name_ and 'fc' not in name_:
                    p.requires_grad_(False)
        m.fc = nn.Sequential(nn.Dropout(0.2), nn.Linear(2048, num_classes))
    elif name == 'EfficientNetB0':
        m = tv_models.efficientnet_b0(weights=tv_models.EfficientNet_B0_Weights.DEFAULT)
        if freeze_backbone:
            for p in m.features.parameters(): p.requires_grad_(False)
        m.classifier = nn.Sequential(
            nn.Dropout(0.2),
            nn.Linear(1280, num_classes)
        )
    else:
        raise ValueError(f'Unknown model: {name}')
    return m


# ── Quick parameter count ──────────────────────────────────────────────────
def count_params(model):
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

dpa = DPA_EfficientNet()
t, tr = count_params(dpa)
print(f'DPA-EfficientNet  |  Total: {t/1e6:.2f}M  |  Trainable: {tr/1e6:.2f}M')
for name in ['MobileNetV2','ResNet50','EfficientNetB0']:
    b = build_baseline(name)
    t, tr = count_params(b)
    print(f'{name:<16}  |  Total: {t/1e6:.2f}M  |  Trainable: {tr/1e6:.2f}M')

print('\n✅ All model architectures defined.')

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision.models as tv_models
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.cuda.amp import GradScaler, autocast

import pandas as pd, numpy as np, cv2, os, glob, time, copy
import matplotlib.pyplot as plt, matplotlib.patches as mpatches
import seaborn as sns
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, roc_auc_score, roc_curve, auc
)
from sklearn.preprocessing import label_binarize
from sklearn.utils.class_weight import compute_class_weight

# ── Global configuration  ──────────────────────────────────────────────────────────
SEED        = 42
IMG_SIZE    = 224
BATCH_SIZE  = 32
NUM_CLASSES = 7
DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

CLASS_NAMES = ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']
CLASS_FULL  = [
    'Actinic Keratosis', 'Basal Cell Carcinoma', 'Benign Keratosis',
    'Dermatofibroma', 'Melanoma', 'Melanocytic Nevi', 'Vascular Lesion'
]

torch.manual_seed(SEED)
np.random.seed(SEED)
print(f'✅ Config ready  |  Device: {DEVICE}  |  Classes: {NUM_CLASSES}')
# ══════════════════════════════════════════════════════════════════════════
#  Reusable training engine — Section 3.6
#  Adam + weight decay + ReduceLROnPlateau + AMP + EarlyStopping
# ══════════════════════════════════════════════════════════════════════════n
def train_one_epoch(model, loader, optimizer, criterion, scaler, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        with autocast():
            logits = model(imgs)
            loss   = criterion(logits, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item() * imgs.size(0)
        correct      += (logits.argmax(1) == labels).sum().item()
        total        += imgs.size(0)
    return running_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    all_probs, all_labels = [], []
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        logits = model(imgs)
        loss   = criterion(logits, labels)
        running_loss += loss.item() * imgs.size(0)
        probs  = torch.softmax(logits, dim=1)
        correct += (logits.argmax(1) == labels).sum().item()
        total   += imgs.size(0)
        all_probs.append(probs.cpu().numpy())
        all_labels.append(labels.cpu().numpy())
    all_probs  = np.vstack(all_probs)
    all_labels = np.concatenate(all_labels)
    y_bin = label_binarize(all_labels, classes=list(range(NUM_CLASSES)))
    auc_val = roc_auc_score(y_bin, all_probs, multi_class='ovr', average='macro')
    return running_loss / total, correct / total, auc_val, all_probs, all_labels


def fit_model(model, train_loader, val_loader, model_name,
              epochs=30, lr=1e-3, wd=1e-4, patience=7, device=DEVICE):
    """
    Full training loop with:
      - Cross-entropy loss (Eq. 10)
      - Adam + weight decay
      - ReduceLROnPlateau on val_auc
      - Early stopping on val_auc
      - Mixed-precision (AMP)
    Returns best model weights + history dict.
    """
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()      # Eq. 10
    optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()),
                                 lr=lr, weight_decay=wd)
    scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5,
                                  patience=3)
    scaler    = GradScaler()

    history = {k: [] for k in
               ['train_loss','train_acc','val_loss','val_acc','val_auc','lr']}
    best_auc, best_wts, no_improve = 0.0, None, 0

    print(f'\n{"="*55}')
    print(f'  Training: {model_name}')
    print(f'  Epochs: {epochs}  |  LR: {lr}  |  Patience: {patience}')
    print(f'{"="*55}')

    for epoch in range(1, epochs + 1):
        t0 = time.time()
        tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer,
                                          criterion, scaler, device)
        vl_loss, vl_acc, vl_auc, _, _ = evaluate(model, val_loader,
                                                   criterion, device)
        scheduler.step(vl_auc)

        history['train_loss'].append(tr_loss)
        history['train_acc'].append(tr_acc)
        history['val_loss'].append(vl_loss)
        history['val_acc'].append(vl_acc)
        history['val_auc'].append(vl_auc)
        history['lr'].append(optimizer.param_groups[0]['lr'])

        print(f'Ep {epoch:03d}/{epochs} | '
              f'Loss: {tr_loss:.4f}/{vl_loss:.4f} | '
              f'Acc: {tr_acc:.4f}/{vl_acc:.4f} | '
              f'AUC: {vl_auc:.4f} | '
              f'{time.time()-t0:.1f}s')

        if vl_auc > best_auc:
            best_auc  = vl_auc
            best_wts  = copy.deepcopy(model.state_dict())
            no_improve = 0
            torch.save(best_wts, f'{model_name}_best.pt')
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f'  Early stopping at epoch {epoch}')
                break

    model.load_state_dict(best_wts)
    print(f'\n  ✅ {model_name} best val AUC: {best_auc:.4f}')
    return model, history, best_auc


print('✅ Training engine ready.')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  DPA-EfficientNet — Two-phase training
#  Phase 1: Backbone frozen, train DPA head only  (15 epochs, lr=1e-3)
#  Phase 2: Unfreeze top-30 layers, fine-tune     (25 epochs, lr=1e-4)
# ══════════════════════════════════════════════════════════════════════════

dpa_model = DPA_EfficientNet(num_classes=NUM_CLASSES).to(DEVICE)

# ── Phase 1: freeze backbone ───────────────────────────────────────────────
for p in dpa_model.features.parameters():
    p.requires_grad_(False)

tr_params = sum(p.numel() for p in dpa_model.parameters() if p.requires_grad)
print(f'Phase 1 trainable params: {tr_params/1e6:.2f}M')

dpa_model, hist_p1, _ = fit_model(
    dpa_model, train_loader, val_loader,
    model_name='DPA_Phase1',
    epochs=15, lr=1e-3, wd=1e-4, patience=5
)

# ── Phase 2: unfreeze top-30 backbone layers ───────────────────────────────
backbone_layers = list(dpa_model.features.children())
for layer in backbone_layers[:-3]:                    # freeze all but last 3 blocks
    for p in layer.parameters():
        p.requires_grad_(False)
for layer in backbone_layers[-3:]:                    # unfreeze last 3 blocks
    for p in layer.parameters():
        p.requires_grad_(True)

tr_params = sum(p.numel() for p in dpa_model.parameters() if p.requires_grad)
print(f'\nPhase 2 trainable params: {tr_params/1e6:.2f}M')

dpa_model, hist_p2, dpa_best_auc = fit_model(
    dpa_model, train_loader, val_loader,
    model_name='DPA_EfficientNet',
    epochs=25, lr=1e-4, wd=1e-5, patience=7
)

# Combine phase histories
def merge_hist(h1, h2):
    return {k: h1[k] + h2[k] for k in h1}

dpa_history = merge_hist(hist_p1, hist_p2)
phase1_end  = len(hist_p1['train_loss'])

print(f'\n✅ DPA-EfficientNet best val AUC: {dpa_best_auc:.4f}')

In [ ]:
import matplotlib.pyplot as plt
import os

# ── Check for required variables ─────────────────────────────────────────
# This ensures the cell doesn't crash if training hasn't completed yet.
if 'dpa_history' not in globals():
    print("❌ Error: 'dpa_history' not found.")
    print("Please ensure you have executed the DPA-EfficientNet training cell (BYbE6wJK3g-d) first.")
else:
    # ── Plot training curves for DPA-EfficientNet (Figure in paper) ───────────
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    fig.suptitle('DPA-EfficientNet — Training Curves (Phase 1 + Phase 2)',
                 fontsize=13, fontweight='bold')

    metrics_to_plot = [('train_loss','val_loss','Loss'),
                       ('train_acc', 'val_acc', 'Accuracy'),
                       ('val_auc',   None,      'Val AUC')]

    # Use phase1_end if defined, otherwise default to 0
    p1_end = globals().get('phase1_end', 0)

    for ax, (tr_key, vl_key, title) in zip(axes, metrics_to_plot):
        epochs_x = range(1, len(dpa_history[tr_key]) + 1)
        ax.plot(epochs_x, dpa_history[tr_key], 'b-', lw=1.5, label='Train')
        if vl_key:
            ax.plot(epochs_x, dpa_history[vl_key], 'r-', lw=1.5, label='Val')
        else:
            # For AUC, we usually only have val_auc
            ax.plot(epochs_x, dpa_history[tr_key], 'g-', lw=2, label='Val AUC')

        if p1_end > 0:
            ax.axvline(p1_end, color='gray', ls='--', alpha=0.7, label='Phase 2 │')

        ax.set_title(title, fontweight='bold')
        ax.set_xlabel('Epoch')
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig('dpa_training_curves.png', dpi=150, bbox_inches='tight')
    plt.show()

    # ── Compare AUC curves across all models ─────────────────────────────────
    fig, ax = plt.subplots(figsize=(10, 5))
    colors_map = {'MobileNetV2':'#e74c3c','ResNet50':'#f39c12',
                  'EfficientNetB0':'#3498db','DPA-EfficientNet':'#2ecc71'}

    # Define baseline_histories if it's not already defined
    if 'baseline_histories' not in locals() and 'baseline_histories' not in globals():
        baseline_histories = {}

    comparison_dict = {**baseline_histories, 'DPA-EfficientNet': dpa_history}

    for name, hist in comparison_dict.items():
        if 'val_auc' in hist and len(hist['val_auc']) > 0:
            lw = 3 if name == 'DPA-EfficientNet' else 1.5
            ax.plot(range(1, len(hist['val_auc'])+1), hist['val_auc'],
                    lw=lw, label=name, color=colors_map.get(name,'gray'))

    ax.set_title('Validation AUC Comparison — All Models', fontweight='bold', fontsize=13)
    ax.set_xlabel('Epoch'); ax.set_ylabel('Val AUC')
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig('auc_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import torchvision.models as tv_models
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import time, copy, cv2, os, glob
from PIL import Image
from torch.cuda.amp import GradScaler, autocast
from torch.optim.lr_scheduler import ReduceLROnPlateau
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.preprocessing import label_binarize
from sklearn.utils.class_weight import compute_class_weight

# ── Global configuration ──
NUM_CLASSES = 7
DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CLASS_NAMES = ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']
IMG_SIZE    = 224
BATCH_SIZE  = 32

# ── Data Loading & Preprocessing ──
def get_loaders():
    # Broad search for the metadata file in case of nested unzipping
    search_csv = glob.glob('**/HAM10000_metadata.csv', recursive=True)
    if not search_csv:
        raise FileNotFoundError("Could not find HAM10000_metadata.csv. Please check the dataset folder.")
    csv_path = search_csv[0]
    print(f"✅ Found metadata at: {csv_path}")

    df = pd.read_csv(csv_path)

    # Map image_id to full path by searching all subdirectories for .jpg files
    image_files = glob.glob('**/*.jpg', recursive=True)
    path_map = {os.path.splitext(os.path.basename(p))[0]: p for p in image_files}

    df['path'] = df['image_id'].map(path_map)
    df = df.dropna(subset=['path']).reset_index(drop=True)
    df['label'] = df['dx'].map({c: i for i, c in enumerate(CLASS_NAMES)})

    train_df = df.sample(frac=0.8, random_state=42)
    val_df = df.drop(train_df.index)

    class HAMDataset(Dataset):
        def __init__(self, df, transform=None):
            self.df = df.reset_index(drop=True)
            self.transform = transform
        def __len__(self): return len(self.df)
        def __getitem__(self, idx):
            row = self.df.iloc[idx]
            img = Image.open(row['path']).convert('RGB')
            if self.transform: img = self.transform(img)
            return img, int(row['label'])

    train_tf = transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.RandomHorizontalFlip(), transforms.ToTensor(), transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
    val_tf = transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.ToTensor(), transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])

    train_ds = HAMDataset(train_df, train_tf)
    val_ds = HAMDataset(val_df, val_tf)

    labels = train_df['label'].values
    weights = compute_class_weight('balanced', classes=np.unique(labels), y=labels)
    samples_weight = torch.tensor([weights[l] for l in labels])
    sampler = WeightedRandomSampler(samples_weight, len(samples_weight))

    t_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=2)
    v_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    return t_loader, v_loader

train_loader, val_loader = get_loaders()

def train_one_epoch(model, loader, optimizer, criterion, scaler, device):
    model.train()
    rl, c, t = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        with autocast():
            logits = model(imgs)
            loss = criterion(logits, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        rl += loss.item() * imgs.size(0)
        c += (logits.argmax(1) == labels).sum().item()
        t += imgs.size(0)
    return rl / t, c / t

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    rl, c, t = 0.0, 0, 0
    ap, al = [], []
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        logits = model(imgs)
        loss = criterion(logits, labels)
        rl += loss.item() * imgs.size(0)
        probs = torch.softmax(logits, dim=1)
        c += (logits.argmax(1) == labels).sum().item()
        t += imgs.size(0)
        ap.append(probs.cpu().numpy())
        al.append(labels.cpu().numpy())
    ap, al = np.vstack(ap), np.concatenate(al)
    y_bin = label_binarize(al, classes=list(range(NUM_CLASSES)))
    auc_v = roc_auc_score(y_bin, ap, multi_class='ovr', average='macro')
    return rl / t, c / t, auc_v, ap, al

def fit_model(model, t_loader, v_loader, model_name, epochs=5):
    model = model.to(DEVICE)
    crit = nn.CrossEntropyLoss()
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    sched = ReduceLROnPlateau(opt, mode='max', factor=0.5, patience=2)
    scaler = GradScaler()
    hist = {'val_auc': []}
    best_auc = 0.0
    for epoch in range(1, epochs + 1):
        tr_loss, tr_acc = train_one_epoch(model, t_loader, opt, crit, scaler, DEVICE)
        vl_loss, vl_acc, vl_auc, _, _ = evaluate(model, v_loader, crit, DEVICE)
        sched.step(vl_auc)
        hist['val_auc'].append(vl_auc)
        print(f'[{model_name}] Ep {epoch} | AUC: {vl_auc:.4f}')
        if vl_auc > best_auc: best_auc = vl_auc
    return model, hist, best_auc

def build_baseline(name):
    if name == 'MobileNetV2': m = tv_models.mobilenet_v2(weights='DEFAULT'); m.classifier[1] = nn.Linear(m.last_channel, NUM_CLASSES)
    elif name == 'ResNet50': m = tv_models.resnet50(weights='DEFAULT'); m.fc = nn.Linear(2048, NUM_CLASSES)
    elif name == 'EfficientNetB0': m = tv_models.efficientnet_b0(weights='DEFAULT'); m.classifier[1] = nn.Linear(1280, NUM_CLASSES)
    return m

baseline_histories = {}
for name in ['MobileNetV2', 'ResNet50', 'EfficientNetB0']:
    print(f'Training {name}...')
    _, hist, _ = fit_model(build_baseline(name), train_loader, val_loader, name)
    baseline_histories[name] = hist

In [ ]:
# ── Confusion matrix for DPA-EfficientNet ─────────────────────────────────
dpa_m   = all_model_metrics['DPA-EfficientNet']
y_true  = dpa_m['labels']
y_pred  = dpa_m['preds']
cm      = confusion_matrix(y_true, y_pred)
acc     = accuracy_score(y_true, y_pred)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            linewidths=0.4, linecolor='#7F8C8D', ax=ax, annot_kws={'size':13})
ax.set_ylabel('True Label', fontweight='bold', fontsize=12)
ax.set_xlabel('Predicted Label', fontweight='bold', fontsize=12)
ax.set_title(f'DPA-EfficientNet Confusion Matrix\nAccuracy: {acc*100:.2f}%',
             fontweight='bold', fontsize=14, pad=12)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print(f'✅ Confusion matrix saved  |  Accuracy: {acc*100:.2f}%')

In [ ]:
# ── Per-class ROC for DPA-EfficientNet ────────────────────────────────────
probs  = dpa_m['probs']
y_bin  = label_binarize(y_true, classes=list(range(NUM_CLASSES)))

fig, ax = plt.subplots(figsize=(10, 7))
pal = ['#e74c3c','#3498db','#2ecc71','#f39c12','#9b59b6','#1abc9c','#e67e22']

for i, (cls, color) in enumerate(zip(CLASS_NAMES, pal)):
    fpr, tpr, _ = roc_curve(y_bin[:, i], probs[:, i])
    roc_auc = auc(fpr, tpr)
    lw = 3 if cls == 'mel' else 1.5
    ax.plot(fpr, tpr, color=color, lw=lw,
            label=f'{cls} (AUC={roc_auc:.3f}){" ← Melanoma" if cls=="mel" else ""}')

ax.plot([0,1],[0,1],'k--',lw=1)
ax.set_xlim([0,1]); ax.set_ylim([0,1.02])
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('DPA-EfficientNet — Per-Class ROC Curves', fontweight='bold', fontsize=13)
ax.legend(loc='lower right', fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('roc_dpa.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Melanoma ROC: all models on one plot ──────────────────────────────────
mel_idx = CLASS_NAMES.index('mel')
fig, ax = plt.subplots(figsize=(8, 6))

for name, m in all_model_metrics.items():
    y_b = label_binarize(m['labels'], classes=list(range(NUM_CLASSES)))
    fpr, tpr, _ = roc_curve(y_b[:, mel_idx], m['probs'][:, mel_idx])
    roc_auc = auc(fpr, tpr)
    lw = 3 if name == 'DPA-EfficientNet' else 1.5
    ax.plot(fpr, tpr, lw=lw,
            color=colors_map.get(name,'gray'),
            label=f'{name} (AUC={roc_auc:.3f})')

ax.plot([0,1],[0,1],'k--',lw=1)
ax.set_title('Melanoma ROC — All Models', fontweight='bold', fontsize=13)
ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
ax.legend(fontsize=10); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('roc_melanoma_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Side-by-side bar chart — all metrics, all models ─────────────────────
model_names_plot = list(all_model_metrics.keys())
metrics_plot     = ['acc', 'mel_prec', 'mel_recall', 'mel_f1', 'auc']
metric_labels    = ['Accuracy', 'Precision\n(Melanoma)', 'Recall\n(Melanoma)',
                    'F1-Score\n(Melanoma)', 'AUC']

x      = np.arange(len(metrics_plot))
width  = 0.18
pal_b  = ['#e74c3c','#f39c12','#3498db','#2ecc71']

fig, ax = plt.subplots(figsize=(14, 6))

for i, (name, color) in enumerate(zip(model_names_plot, pal_b)):
    vals = [all_model_metrics[name][m] for m in metrics_plot]
    bars = ax.bar(x + i*width, vals, width, label=name, color=color, alpha=0.85)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{val:.3f}', ha='center', va='bottom', fontsize=8, rotation=45)

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(metric_labels, fontsize=11)
ax.set_ylim(0, 1.12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Model Comparison — Table 2 Visualised', fontweight='bold', fontsize=13)
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)
ax.axhline(y=1.0, color='gray', ls='--', lw=0.8)
plt.tight_layout()
plt.savefig('model_comparison_bar.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  Grad-CAM++ on validation samples — no upload needed
#  Automatically pulls real val images from the DataLoader
# ══════════════════════════════════════════════════════════════════════════

class GradCAMPlusPlus:
    """
    Section 3.7 — Grad-CAM++ with higher-order gradients.
    Hooks into target layer to compute class-specific importance weights.
    """
    def __init__(self, model, target_layer):
        self.model       = model
        self.gradients   = None
        self.activations = None
        self._hooks = [
            target_layer.register_forward_hook(self._fwd_hook),
            target_layer.register_backward_hook(self._bwd_hook),
        ]

    def _fwd_hook(self, module, inp, out): self.activations = out
    def _bwd_hook(self, module, gi, go):  self.gradients   = go[0]
    def remove_hooks(self):
        for h in self._hooks: h.remove()

    def __call__(self, x, class_idx=None):
        self.model.eval()
        logits = self.model(x)
        if class_idx is None:
            class_idx = logits.argmax(dim=1).item()
        self.model.zero_grad()
        logits[0, class_idx].backward()

        grads = self.gradients    # (1, C, H, W)
        acts  = self.activations  # (1, C, H, W)

        alpha_num   = grads.pow(2)
        alpha_denom = 2*grads.pow(2) + (acts*grads.pow(3)).sum(dim=(2,3), keepdim=True)
        alpha_denom = torch.where(alpha_denom != 0, alpha_denom,
                                   torch.ones_like(alpha_denom))
        alphas  = alpha_num / alpha_denom
        weights = (F.relu(grads) * alphas).sum(dim=(2,3), keepdim=True)
        cam     = F.relu((weights * acts).sum(dim=1, keepdim=True))

        cam = cam.squeeze().detach().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam, class_idx, logits.detach().cpu()


def overlay_cam(cam, img_tensor, alpha=0.4):
    """Blend Grad-CAM++ heatmap onto the original image."""
    # Denormalize
    mean = torch.tensor(MEAN).view(3,1,1)
    std  = torch.tensor(STD).view(3,1,1)
    img_np = (img_tensor.squeeze().cpu() * std + mean).permute(1,2,0).numpy()
    img_np = np.clip(img_np, 0, 1)

    cam_resized = cv2.resize(cam, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_CUBIC)
    cam_resized = cv2.GaussianBlur(cam_resized, (7,7), 0)
    heatmap     = cv2.applyColorMap(np.uint8(255*cam_resized), cv2.COLORMAP_JET)
    heatmap_rgb = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB) / 255.0
    overlay     = np.clip((1-alpha)*img_np + alpha*heatmap_rgb, 0, 1)
    return img_np, cam_resized, overlay


# ── Pull a batch from val loader & visualise ──────────────────────────────
target_layer = dpa_model.features[-1]   # Last conv block of EfficientNet-B0
cam_engine   = GradCAMPlusPlus(dpa_model, target_layer)

# Get one batch
imgs_batch, labels_batch = next(iter(val_loader))
N_SHOW = 6

fig, axes = plt.subplots(N_SHOW, 3, figsize=(12, N_SHOW*3.2))
fig.suptitle('Grad-CAM++ Visualisation — DPA-EfficientNet (Section 3.7)',
             fontsize=14, fontweight='bold', y=1.01)

for i in range(N_SHOW):
    inp   = imgs_batch[i].unsqueeze(0).to(DEVICE).requires_grad_(True)
    true_idx = labels_batch[i].item()

    cam, pred_idx, logits = cam_engine(inp)
    probs      = torch.softmax(logits[0], dim=0).numpy()
    confidence = probs[pred_idx] * 100
    pred_name  = CLASS_NAMES[pred_idx]
    true_name  = CLASS_NAMES[true_idx]
    correct    = '✅' if pred_idx == true_idx else '❌'

    img_orig, cam_map, overlay = overlay_cam(cam, imgs_batch[i])

    axes[i,0].imshow(img_orig)
    axes[i,0].set_title(f'Original\nTrue: {true_name}', fontsize=9)
    axes[i,0].axis('off')

    axes[i,1].imshow(cam_map, cmap='jet', vmin=0, vmax=1)
    axes[i,1].set_title('Grad-CAM++ Heatmap', fontsize=9)
    axes[i,1].axis('off')

    axes[i,2].imshow(overlay)
    axes[i,2].set_title(f'Overlay {correct}\nPred: {pred_name} ({confidence:.1f}%)', fontsize=9)
    axes[i,2].axis('off')

cam_engine.remove_hooks()
plt.tight_layout()
plt.savefig('gradcampp_figure5.png', dpi=200, bbox_inches='tight', facecolor='white')
plt.show()
print('✅ Grad-CAM++ figure saved as gradcampp_figure5.png')

In [ ]:
# ── Save model ────────────────────────────────────────────────────────────
torch.save(dpa_model.state_dict(), 'DPA_EfficientNet_final.pt')
print('✅ Model weights saved → DPA_EfficientNet_final.pt')

# ── Final results table ───────────────────────────────────────────────────
dpa_res = all_model_metrics['DPA-EfficientNet']

print('\n' + '='*55)
print('  DPA-EfficientNet — Final Results')
print('='*55)
print(f'  Accuracy          : {dpa_res["acc"]*100:.2f}%')
print(f'  AUC (macro OvR)   : {dpa_res["auc"]:.4f}')
print(f'  Melanoma Precision : {dpa_res["mel_prec"]:.4f}')
print(f'  Melanoma Recall   : {dpa_res["mel_recall"]:.4f}')
print(f'  Melanoma F1       : {dpa_res["mel_f1"]:.4f}')
print('='*55)
print('  Paper targets (Table 2):')
print('  Acc: 94.2% | AUC: 0.97 | Rec(mel): 0.89')
print('='*55)

# ── Download all saved files ──────────────────────────────────────────────
from google.colab import files
output_files = [
    'DPA_EfficientNet_final.pt',
    'confusion_matrix.png',
    'roc_dpa.png',
    'roc_melanoma_comparison.png',
    'dpa_training_curves.png',
    'auc_comparison.png',
    'model_comparison_bar.png',
    'gradcampp_figure5.png',
    'class_distribution.png',
]
for fname in output_files:
    if os.path.exists(fname):
        files.download(fname)
        print(f'  ↓ {fname}')

print('\n🎉 Pipeline complete!')